# 知识图谱真实库回放与质量基线

这份 notebook 用来固定跑一轮真实库质量基线：先写入一组受控测试数据，再跑 `kg incremental-refresh` 等价应用流程、`rebuild-wiki`、`rebuild-indexes`，最后用一组有特点的检索 bad case 回放。

目标不是模拟单测，而是验证真实 Postgres、真实 Milvus、真实 embedding、真实 LLM/tmux backend 下，知识图谱从入库到召回的完整链路是否稳定。

## 0. 运行参数

`TARGET="prod"` 会写真实库。下面默认就是写真实库，因为这份脚本用于真实回放。若只想检查 record 结构，把 `DRY_RUN=True`。

In [ ]:
from __future__ import annotations

import json
import logging
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "smart-fund-server" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name != "smart-fund-server":
    candidate = Path.cwd() / ".claude" / "skills" / "smart-fund-server"
    if candidate.exists():
        PROJECT_ROOT = candidate.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)
logging.getLogger("src.domain.knowledge.compiler").setLevel(logging.INFO)
logging.getLogger("src.infrastructure.llm_proxy.service").setLevel(logging.INFO)
logging.getLogger("src.infrastructure.vector_store").setLevel(logging.INFO)

TARGET = "prod"
ADAPTER = "financial"
DRY_RUN = False

# 真实业务增量刷新会读取 ft_* 并触发新闻 LLM 抽取。若只验证受控数据闭环，可改成 False。
RUN_REAL_INCREMENTAL_REFRESH = True
INCREMENTAL_CODES = ["300750", "603305"]
INCREMENTAL_STOCK_LIMIT = 50
INCREMENTAL_NEWS_LIMIT = 5
COMPILE_CONCURRENCY = 2

BASELINE_TS = "2026-04-29T09:30:00+08:00"
BASELINE_CASE_FILE = PROJECT_ROOT / "docs/6. 使用说明/知识图谱/generated_real_replay_bad_cases.json"

{
    "project_root": str(PROJECT_ROOT),
    "target": TARGET,
    "adapter": ADAPTER,
    "dry_run": DRY_RUN,
    "run_real_incremental_refresh": RUN_REAL_INCREMENTAL_REFRESH,
    "baseline_case_file": str(BASELINE_CASE_FILE),
}


## 1. 初始化服务并强制检查 Milvus

当前设计里 Milvus 是研究上下文召回的强依赖。这里先检查配置和客户端可用性，避免后面 replay 出现假通过。

In [ ]:
from sqlalchemy import text

from src.application.dto.knowledge_dto import (
    KnowledgeBadCaseReplayCommand,
    KnowledgeCompileCommand,
    KnowledgeIncrementalRefreshCommand,
    KnowledgeRebuildIndexesCommand,
    KnowledgeRebuildWikiCommand,
    KnowledgeResearchContextBadCase,
    KnowledgeResearchContextCommand,
)
from src.application.services.knowledge_service import create_knowledge_service
from src.infrastructure.config import settings
from src.infrastructure.connections import get_session
from src.infrastructure.vector_store.milvus_hybrid_store import MilvusHybridStore

assert settings.MILVUS_ENABLED is True, "Milvus 必须强制启用，不能用环境变量关闭"
MilvusHybridStore().ensure_ready()

service = create_knowledge_service(target=TARGET)
health = await service.health()
pprint(health.to_dict())
assert health.status == "ok", health.to_dict()


## 2. 构造受控基线数据

这批数据专门覆盖五类问题：

- 证券代码和公司名双入口：`宁德时代 / 300750`
- 事件到行业/概念影响：并购重组影响哪些行业
- 宏观因子到资产/风格：低利率利好成长资产
- 地缘事件到大类资产：中东冲突影响原油、黄金、航空
- 语义泛化：查询词不完全等于原文，比如“海外工厂投产”召回“海外产能扩张”

In [ ]:
def stock_entity(code: str, name: str, exchange: str = "SZ", confidence: float = 0.95) -> dict:
    return {
        "type": "stock",
        "exchange": exchange,
        "code": code,
        "name": name,
        "aliases": [code, f"{code}.{exchange}"],
        "confidence": confidence,
    }


def concept(name: str, *, taxonomy: str = "baseline", confidence: float = 0.86, **extra) -> dict:
    return {"type": "concept", "taxonomy": taxonomy, "name": name, "confidence": confidence, **extra}


def industry(name: str, *, taxonomy: str = "baseline", confidence: float = 0.86, **extra) -> dict:
    return {"type": "industry", "taxonomy": taxonomy, "name": name, "confidence": confidence, **extra}


def macro_indicator(code: str, name: str, confidence: float = 0.9) -> dict:
    return {
        "type": "macro_indicator",
        "indicator_code": code,
        "name": name,
        "confidence": confidence,
    }


records = [
    {
        "source_type": "stock_basics",
        "source_id": "notebook_baseline:stock:300750",
        "observed_at": BASELINE_TS,
        "payload": {
            "source_id": "notebook_baseline:stock:300750",
            "exchange": "SZ",
            "code": "300750",
            "name": "宁德时代",
            "company_name": "宁德时代新能源科技股份有限公司",
            "aliases": ["CATL", "300750", "300750.SZ"],
            "status": "active",
        },
    },
    {
        "source_type": "stock_basics",
        "source_id": "notebook_baseline:stock:603305",
        "observed_at": BASELINE_TS,
        "payload": {
            "source_id": "notebook_baseline:stock:603305",
            "exchange": "SH",
            "code": "603305",
            "name": "旭升集团",
            "aliases": ["603305", "603305.SH"],
            "status": "active",
        },
    },
    {
        "source_type": "news_articles",
        "source_id": "notebook_baseline:news:catl_overseas_capacity",
        "observed_at": BASELINE_TS,
        "payload": {
            "source_id": "notebook_baseline:news:catl_overseas_capacity",
            "document_id": "notebook_baseline:news:catl_overseas_capacity",
            "title": "宁德时代海外产能扩张带动储能供应链订单",
            "published_at": BASELINE_TS,
            "source_name": "Notebook基线",
            "text": "宁德时代推进欧洲和东南亚海外产能扩张，储能电芯、快充电池和新能源车产业链订单预期改善。市场认为海外工厂投产有助于降低贸易壁垒，并改善供应链交付能力。",
            "mentioned_entities": [
                stock_entity("300750", "宁德时代"),
                concept("海外产能"),
                concept("快充"),
                industry("储能产业链"),
            ],
            "affected_entities": [
                {**stock_entity("300750", "宁德时代"), "direction": "positive", "reason": "海外产能扩张改善交付能力"},
                {**industry("储能产业链"), "direction": "positive", "reason": "储能电芯订单预期改善"},
                {**industry("新能源车产业链"), "direction": "positive", "reason": "快充和动力电池需求提升"},
            ],
        },
    },
    {
        "source_type": "news_articles",
        "source_id": "notebook_baseline:news:ma_industry_rotation",
        "observed_at": BASELINE_TS,
        "payload": {
            "source_id": "notebook_baseline:news:ma_industry_rotation",
            "document_id": "notebook_baseline:news:ma_industry_rotation",
            "title": "并购重组政策活跃提升券商、半导体设备和新能源车风险偏好",
            "published_at": BASELINE_TS,
            "source_name": "Notebook基线",
            "text": "资本市场并购重组审核提速，市场预期产业整合会带动券商投行业务、半导体设备国产替代和新能源车产业链估值修复。并购重组主题也可能提升中小市值公司的交易活跃度。",
            "mentioned_entities": [
                concept("并购重组"),
                industry("券商"),
                industry("半导体设备"),
                industry("新能源车产业链"),
            ],
            "affected_entities": [
                {**industry("券商"), "direction": "positive", "reason": "投行业务弹性提升"},
                {**industry("半导体设备"), "direction": "positive", "reason": "产业整合和国产替代预期加强"},
                {**industry("新能源车产业链"), "direction": "positive", "reason": "估值修复和整合预期"},
                {**concept("中小市值"), "direction": "positive", "reason": "交易活跃度提升"},
            ],
        },
    },
    {
        "source_type": "policy_news",
        "source_id": "notebook_baseline:policy:low_rate_growth_assets",
        "observed_at": BASELINE_TS,
        "payload": {
            "source_id": "notebook_baseline:policy:low_rate_growth_assets",
            "document_id": "notebook_baseline:policy:low_rate_growth_assets",
            "title": "低利率环境和资本市场改革提升成长资产估值",
            "published_at": BASELINE_TS,
            "source_name": "Notebook基线",
            "text": "低利率环境降低权益资产折现率，资本市场改革改善风险偏好。成长资产、科技创新、半导体设备和新能源车产业链可能受益，但高股息资产的相对吸引力可能下降。",
            "mentioned_entities": [
                macro_indicator("low_rate_environment", "低利率环境"),
                concept("资本市场改革"),
                concept("成长资产"),
            ],
            "affected_entities": [
                {**concept("成长资产"), "direction": "positive", "reason": "折现率下降提升估值"},
                {**industry("半导体设备"), "direction": "positive", "reason": "成长风格风险偏好改善"},
                {**industry("新能源车产业链"), "direction": "positive", "reason": "成长股估值弹性提升"},
                {**concept("高股息资产"), "direction": "negative", "reason": "相对吸引力下降"},
            ],
        },
    },
    {
        "source_type": "news_articles",
        "source_id": "notebook_baseline:news:middle_east_assets",
        "observed_at": BASELINE_TS,
        "payload": {
            "source_id": "notebook_baseline:news:middle_east_assets",
            "document_id": "notebook_baseline:news:middle_east_assets",
            "title": "中东冲突升温推升原油和黄金避险需求",
            "published_at": BASELINE_TS,
            "source_name": "Notebook基线",
            "text": "中东冲突升温可能扰动能源运输通道，原油价格和黄金避险需求上升。航空运输行业面临燃油成本压力，化工产业链也会受到原油成本传导影响。",
            "mentioned_entities": [
                concept("中东冲突"),
                {"type": "commodity", "name": "原油", "confidence": 0.88},
                {"type": "commodity", "name": "黄金", "confidence": 0.88},
                industry("航空运输"),
            ],
            "affected_entities": [
                {"type": "commodity", "name": "原油", "direction": "positive", "confidence": 0.88, "reason": "供应扰动和运输通道风险"},
                {"type": "commodity", "name": "黄金", "direction": "positive", "confidence": 0.86, "reason": "避险需求上升"},
                {**industry("航空运输"), "direction": "negative", "reason": "燃油成本压力上升"},
                {**industry("化工产业链"), "direction": "negative", "reason": "原油成本向下游传导"},
            ],
        },
    },
    {
        "source_type": "derived_signal",
        "source_id": "notebook_baseline:signal:catl_flow",
        "observed_at": BASELINE_TS,
        "payload": {
            "source_id": "notebook_baseline:signal:catl_flow",
            "signal_type": "market_flow.stock_net_inflow",
            "observed_at": BASELINE_TS,
            "target_ref": stock_entity("300750", "宁德时代"),
            "title": "宁德时代资金净流入改善",
            "value": 1,
            "unit": "signal",
            "window": "1d",
            "confidence": 0.9,
            "raw_data": {"net_inflow_signal": "positive", "source": "notebook_baseline"},
        },
    },
    {
        "source_type": "derived_signal",
        "source_id": "notebook_baseline:signal:low_rate",
        "observed_at": BASELINE_TS,
        "payload": {
            "source_id": "notebook_baseline:signal:low_rate",
            "signal_type": "macro.low_rate_environment",
            "observed_at": BASELINE_TS,
            "target_ref": macro_indicator("low_rate_environment", "低利率环境"),
            "title": "低利率环境利好成长资产",
            "value": 1,
            "unit": "signal",
            "window": "latest",
            "confidence": 0.9,
            "raw_data": {"rate_signal": "low", "source": "notebook_baseline"},
        },
    },
]

print(f"prepared records: {len(records)}")
pprint([(item["source_type"], item["source_id"]) for item in records])


## 3. 写入受控数据到知识图谱

新闻和政策文本会走 Adapter 内部 LLM enrichment；structured/signal 数据走确定性规则。这里失败要直接停，不进入后续质量回放。

In [ ]:
compile_result = await service.compile_kg(
    KnowledgeCompileCommand(
        adapter_name=ADAPTER,
        target=TARGET,
        records=records,
        dry_run=DRY_RUN,
        concurrency=COMPILE_CONCURRENCY,
    )
)
pprint(compile_result.to_dict())
assert compile_result.failed_records == 0, compile_result.to_dict()
assert compile_result.nodes > 0 and compile_result.evidence > 0, compile_result.to_dict()


## 4. 可选：跑真实业务增量刷新

这一步等价于 `kg incremental-refresh` 的应用服务路径，会读取真实 `ft_*` 数据并补充股票基础信息和相关新闻。为了避免重复 rebuild index，这里让增量刷新先不 rebuild indexes，后面统一重建。

In [ ]:
if RUN_REAL_INCREMENTAL_REFRESH:
    incremental_result = await service.refresh_financial_incremental(
        KnowledgeIncrementalRefreshCommand(
            target=TARGET,
            codes=INCREMENTAL_CODES,
            stock_limit=INCREMENTAL_STOCK_LIMIT,
            news_limit=INCREMENTAL_NEWS_LIMIT,
            dry_run=DRY_RUN,
            concurrency=COMPILE_CONCURRENCY,
            rebuild_indexes=False,
        )
    )
    pprint(incremental_result.to_dict())
    failed_steps = [step for step in incremental_result.steps if step.get("failed_records", 0)]
    assert not failed_steps, failed_steps
else:
    print("skip real incremental refresh")


## 5. 基于真实库重建 Wiki 和 Milvus 索引

`rebuild_indexes` 同时重建图邻接、evidence chunks 和 Milvus hybrid chunks。Milvus 失败应直接暴露，不降级到旧 keyword 检索。

In [ ]:
if DRY_RUN:
    raise RuntimeError("DRY_RUN=True 时不会写 kg_*，不能继续 rebuild wiki/index baseline")

wiki_result = await service.rebuild_wiki_for(
    KnowledgeRebuildWikiCommand(adapter_name=ADAPTER, target=TARGET, scope="all")
)
pprint(wiki_result.to_dict())
assert wiki_result.pages > 0, wiki_result.to_dict()

index_result = await service.rebuild_indexes_for(
    KnowledgeRebuildIndexesCommand(
        adapter_name=ADAPTER,
        target=TARGET,
        scope="all",
        index_types=["graph_adjacency", "evidence_chunks", "hybrid_chunks"],
    )
)
pprint(index_result.to_dict())
assert index_result.graph_adjacency > 0, index_result.to_dict()
assert index_result.evidence_chunks > 0, index_result.to_dict()
assert index_result.hybrid_chunks > 0, index_result.to_dict()


## 6. 固化检索 bad case

这些 case 不要求答案完全一致，但要求最关键的节点、关系、证据和 Milvus 召回通道出现。先跑 deterministic plan 做稳定质量基线，再单独跑 Agentic A-RAG 对比。

In [ ]:
bad_cases = [
    {
        "case_id": "real_baseline_catl_recent_events",
        "query": "宁德时代 300750 最近受哪些事件影响",
        "expected_node_names": ["宁德时代"],
        "expected_relation_types": ["mentions"],
        "expected_channels_used": ["semantic_hybrid_search", "chunk_read"],
        "min_hits": 2,
        "min_evidence_refs": 1,
        "min_matched_nodes": 1,
        "min_matched_edges": 1,
        "retrieval_mode": "deterministic_plan",
    },
    {
        "case_id": "real_baseline_ma_industry_targets",
        "query": "并购重组对哪些行业有影响",
        "expected_node_names": ["并购重组", "券商"],
        "expected_relation_types": ["mentions", "affects"],
        "expected_channels_used": ["semantic_hybrid_search", "chunk_read"],
        "min_hits": 2,
        "min_evidence_refs": 1,
        "min_matched_nodes": 2,
        "min_matched_edges": 1,
        "retrieval_mode": "deterministic_plan",
    },
    {
        "case_id": "real_baseline_low_rate_beneficiaries",
        "query": "低利率环境利好什么资产和行业",
        "expected_node_names": ["低利率环境", "成长资产"],
        "expected_relation_types": ["mentions", "affects"],
        "expected_channels_used": ["semantic_hybrid_search", "chunk_read"],
        "min_hits": 2,
        "min_evidence_refs": 1,
        "min_matched_nodes": 2,
        "min_matched_edges": 1,
        "retrieval_mode": "deterministic_plan",
    },
    {
        "case_id": "real_baseline_middle_east_asset_transmission",
        "query": "中东冲突影响哪些资产和行业",
        "expected_node_names": ["中东冲突", "原油", "黄金"],
        "expected_relation_types": ["mentions", "affects"],
        "expected_channels_used": ["semantic_hybrid_search", "chunk_read"],
        "min_hits": 2,
        "min_evidence_refs": 1,
        "min_matched_nodes": 3,
        "min_matched_edges": 1,
        "retrieval_mode": "deterministic_plan",
    },
    {
        "case_id": "real_baseline_semantic_paraphrase_overseas_factory",
        "query": "海外工厂投产会带动哪些产业链机会",
        "expected_node_names": ["海外产能", "储能产业链"],
        "expected_relation_types": ["mentions"],
        "expected_channels_used": ["semantic_hybrid_search", "chunk_read"],
        "min_hits": 2,
        "min_evidence_refs": 1,
        "min_matched_nodes": 2,
        "min_matched_edges": 1,
        "retrieval_mode": "deterministic_plan",
    },
]

BASELINE_CASE_FILE.write_text(
    json.dumps({"adapter_name": ADAPTER, "target": TARGET, "cases": bad_cases}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"bad cases written: {BASELINE_CASE_FILE}")
pprint(bad_cases)


## 7. 回放 deterministic 质量基线

如果这里失败，先看输出里的 `missing_node_names`、`missing_relation_types`、`missing_channels_used` 和 `metric_failures`。这些字段能直接告诉你是召回通道没跑、Milvus 没召回、图边没连上，还是证据上下文太少。

In [ ]:
deterministic_replay = await service.replay_research_context_bad_cases(
    KnowledgeBadCaseReplayCommand(
        adapter_name=ADAPTER,
        target=TARGET,
        cases=[KnowledgeResearchContextBadCase(**case) for case in bad_cases],
        graph_depth=3,
        graph_limit=30,
        wiki_limit=10,
        evidence_limit=30,
        max_chars=8000,
    )
)

deterministic_data = deterministic_replay.to_dict()
pprint({k: deterministic_data[k] for k in ["total", "passed", "failed", "metrics"]})
for item in deterministic_data["results"]:
    print("\n===", item["case_id"], "===")
    pprint(item)

assert deterministic_replay.failed == 0, deterministic_data


## 8. 对比 Agentic A-RAG 主路径

Agentic 路径允许 LLM 规划检索步骤，适合真实问答主路径；deterministic 路径保留为稳定回放基线。这里同一批 case 再跑一次 Agentic，方便观察是否存在规划偏差。

In [ ]:
def agentic_case_from(case: dict) -> dict:
    # Agentic A-RAG 允许 LLM 自主选择工具链，不能要求每次都固定走 semantic_hybrid_search。
    # 这里仍然检查核心节点、关系和最小召回量，但不把具体 channel 当成硬门禁。
    relaxed = dict(case, retrieval_mode="agentic_arag")
    relaxed["expected_channels_used"] = []
    return relaxed


agentic_cases = [agentic_case_from(case) for case in bad_cases]

agentic_replay = await service.replay_research_context_bad_cases(
    KnowledgeBadCaseReplayCommand(
        adapter_name=ADAPTER,
        target=TARGET,
        cases=[KnowledgeResearchContextBadCase(**case) for case in agentic_cases],
        graph_depth=3,
        graph_limit=30,
        wiki_limit=10,
        evidence_limit=30,
        max_chars=8000,
    )
)

agentic_data = agentic_replay.to_dict()
pprint({k: agentic_data[k] for k in ["total", "passed", "failed", "metrics"]})
for item in agentic_data["results"]:
    print("\n===", item["case_id"], "===")
    pprint({
        "passed": item["passed"],
        "channels_used": item["channels_used"],
        "missing_node_names": item["missing_node_names"],
        "missing_relation_types": item["missing_relation_types"],
        "missing_channels_used": item["missing_channels_used"],
        "metric_failures": item["metric_failures"],
    })

if agentic_replay.failed:
    print("\nAgentic A-RAG 是自由规划路径，当前只作为观测项，不作为稳定回放硬门禁。")
    print("稳定质量门禁以上一节 deterministic replay 为准；若要让 Agentic 也稳定通过，需要继续优化 agent planner / rerank / trace replay。")


## 9. 抽样查看研究上下文

这一步用于人工检查上下文质量：命中了哪些 channel、哪些 node/edge/evidence、最终拼给投研 Agent 的文本是什么。

In [ ]:
sample_queries = [
    "宁德时代 300750 最近受哪些事件影响",
    "并购重组对哪些行业有影响",
    "海外工厂投产会带动哪些产业链机会",
]

for query in sample_queries:
    context = await service.build_research_context_for(
        KnowledgeResearchContextCommand(
            adapter_name=ADAPTER,
            target=TARGET,
            query=query,
            retrieval_mode="agentic_arag",
            graph_depth=3,
            graph_limit=30,
            wiki_limit=10,
            evidence_limit=30,
            max_chars=8000,
        )
    )
    data = context.to_dict()
    print("\n#", query)
    pprint({
        "hits": len(data["hits"]),
        "matched_nodes": len(data["matched_nodes"]),
        "matched_edges": len(data["matched_edges"]),
        "evidence_refs": len(data["evidence_refs"]),
        "channels_used": data["retrieval_channels_used"],
        "milvus_enabled": data["milvus_enabled"],
        "agentic_enabled": data["agentic_enabled"],
    })
    print("--- context_text preview ---")
    print(data["context_text"][:1500])


## 10. 数据库落库抽查

最后确认受控数据确实进入 `kg_evidence`，并且 evidence chunk 已可被索引使用。

In [ ]:
def db_rows(sql: str, params: dict | None = None) -> list[dict]:
    with get_session(TARGET) as session:
        rows = session.execute(text(sql), params or {}).mappings().all()
    return [dict(row) for row in rows]


counts = db_rows(
    """
    select 'kg_nodes' as table_name, count(*) as count from kg_nodes where adapter_name = :adapter
    union all
    select 'kg_edges' as table_name, count(*) as count from kg_edges where adapter_name = :adapter
    union all
    select 'kg_evidence' as table_name, count(*) as count from kg_evidence where adapter_name = :adapter
    union all
    select 'kg_wiki_pages' as table_name, count(*) as count from kg_wiki_pages where adapter_name = :adapter
    union all
    select 'kg_evidence_chunks' as table_name, count(*) as count from kg_evidence_chunks where adapter_name = :adapter
    """,
    {"adapter": ADAPTER},
)
pprint(counts)

baseline_evidence = db_rows(
    """
    select source_type, source_id, left(coalesce(content, ''), 220) as content_preview
    from kg_evidence
    where adapter_name = :adapter and source_id like 'notebook_baseline:%'
    order by source_id
    """,
    {"adapter": ADAPTER},
)
pprint(baseline_evidence)
assert len(baseline_evidence) >= len(records), baseline_evidence
